## **Permisos bajo control: roles seguros para una aplicación interna**

## Introducción

Una empresa necesita asignar permisos dentro de una aplicación interna para solicitudes, aprobaciones y auditorías.

El problema aparece cuando una persona puede tener demasiados permisos, generando riesgos de seguridad.

El objetivo es encontrar una asignación válida de roles para usuarios ficticios cumpliendo diferentes restricciones.

La idea principal del algoritmo es:

**Explorar posibilidades, usar restricciones y retroceder con inteligencia.**

## Modelamiento con Backtracking

| Elemento | En nuestro proyecto |
|---|---|
| Solución parcial | Usuarios que ya tienen un rol asignado |
| Candidatos | Roles que puede recibir el siguiente usuario |
| Restricciones | Reglas de seguridad del sistema |
| Solución completa | Todos los usuarios tienen un rol válido |
| Retroceso | Quitar una asignación incorrecta |
| Poda | Evitar continuar por caminos imposibles |

Ejemplo de solución parcial:

{
"Ana": "editor",
"Luis": "lector"
}

Todavía faltan usuarios por asignar.

In [1]:
usuarios = [
    "Ana",
    "Luis",
    "Maria",
    "Camilo",
    "Sofia"
]

roles = [
    "administrador",
    "editor",
    "aprobador",
    "auditor",
    "lector"
]

print("Usuarios:", usuarios)
print("Roles:", roles)

Usuarios: ['Ana', 'Luis', 'Maria', 'Camilo', 'Sofia']
Roles: ['administrador', 'editor', 'aprobador', 'auditor', 'lector']


In [2]:
estadisticas = {
    "llamadas_recursivas":0,
    "ramas_descartadas":0,
    "retrocesos":0
}

# **Candidatos de Sofía**

In [3]:
def obtener_candidatos(usuario):

    if usuario == "Sofia":
        return ["lector", "auditor"]

    return roles
# Sofia tiene una regla especial, por eso no puede probar todos los roles.

# **Restricciones**



In [4]:
def es_valido(asignacion, usuario, rol):

    # 1. Camilo no puede ser administrador
    if usuario == "Camilo" and rol == "administrador":
        return False

    # 2. Sofia solo puede ser lectora o auditora
    if usuario == "Sofia" and rol not in ["lector", "auditor"]:
        return False

    # 3. Maximo una persona puede ser administrador
    if rol == "administrador":
        if "administrador" in asignacion.values():
            return False

    # 4. Nadie puede ser aprobador y auditor al mismo tiempo
    # Como cada usuario solo recibe un rol, se verifica si se intenta
    # asignar un rol incompatible al mismo usuario
    if usuario in asignacion:
        if asignacion[usuario] == "aprobador" and rol == "auditor":
            return False

        if asignacion[usuario] == "auditor" and rol == "aprobador":
            return False

    # 5. No permitir que se termine una asignación sin auditor
    # (solo se aplica cuando ya están todos los usuarios asignados)
    if len(asignacion) == 4:
        roles_finales = list(asignacion.values()) + [rol]

        if "auditor" not in roles_finales:
            return False

    # 6. No permitir que se termine una asignación sin aprobador
    # (solo se aplica cuando ya están todos los usuarios asignados)
    if len(asignacion) == 4:
        roles_finales = list(asignacion.values()) + [rol]

        if "aprobador" not in roles_finales:
            return False

    return True

# **Solución completa**

In [5]:
def es_solucion_completa(asignacion):

    if len(asignacion) != len(usuarios):
        return False

    roles_asignados = asignacion.values()

    if "auditor" not in roles_asignados:
        return False

    if "aprobador" not in roles_asignados:
        return False

    return True

# **Poda**

In [6]:
def puede_podar(asignacion):

    cantidad_admin = list(asignacion.values()).count(
        "administrador"
    )

    if cantidad_admin > 1:
        return True

    return False

# **Backtracking**

In [7]:
def backtracking(asignacion, posicion=0):

    estadisticas["llamadas_recursivas"] += 1

    # Caso exitoso
    if es_solucion_completa(asignacion):
        return asignacion.copy()

    # Ya no hay usuarios
    if posicion == len(usuarios):
        return None

    usuario = usuarios[posicion]

    for rol in obtener_candidatos(usuario):

        if es_valido(asignacion, usuario, rol):

            # Elegir candidato
            asignacion[usuario] = rol

            if not puede_podar(asignacion):

                resultado = backtracking(
                    asignacion,
                    posicion + 1
                )

                if resultado:
                    return resultado

            else:
                estadisticas["ramas_descartadas"] += 1

            # Retroceso
            del asignacion[usuario]

            estadisticas["retrocesos"] += 1

        else:

            estadisticas["ramas_descartadas"] += 1

    return None

In [8]:
solucion = backtracking({})

print("Asignación encontrada:")
print(solucion)

print("\nEstadísticas:")
print(estadisticas)

Asignación encontrada:
{'Ana': 'administrador', 'Luis': 'editor', 'Maria': 'editor', 'Camilo': 'aprobador', 'Sofia': 'auditor'}

Estadísticas:
{'llamadas_recursivas': 7, 'ramas_descartadas': 6, 'retrocesos': 1}


# Tabla:

In [9]:
import pandas as pd
tabla = pd.DataFrame(
    solucion.items(),
    columns=["Usuario","Rol"]
)

tabla

,Usuario,Rol
0,Ana,administrador
1,Luis,editor
2,Maria,editor
3,Camilo,aprobador
4,Sofia,auditor


# Prueba de caso sin solución

Para comprobar que el algoritmo puede detectar problemas:

Se puede agregar una regla contradictoria:

- Camilo debe ser administrador.
- Camilo no puede ser administrador.

Como ambas reglas no pueden cumplirse al mismo tiempo, el algoritmo debe indicar que no existe solución.

# Conclusión

Este proyecto muestra cómo backtracking permite resolver problemas de asignación.

El algoritmo:

- construye soluciones parciales;
- prueba candidatos;
- verifica restricciones;
- descarta caminos inválidos;
- retrocede cuando una decisión falla.

La poda ayuda a evitar búsquedas innecesarias y mejora la eficiencia.